# 🔷 VIBRO — Manual Training Pipeline

This notebook trains a personalized voice name-detection model for a Vibro user.

**Pipeline:**
1. Download user audio from Supabase
2. Extract MFCC features
3. Train neural network
4. Convert to TFLite
5. Upload model to Supabase

---

## Section A — Environment Setup

In [ ]:
# Install dependencies
!pip install -q supabase tensorflow librosa scikit-learn numpy

In [ ]:
import os
import json
import numpy as np
import librosa
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from supabase import create_client, Client
from datetime import datetime

print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ══════════════════════════════════════════════
# ⚠️  CONFIGURE THESE — DO NOT COMMIT TO GIT
# ══════════════════════════════════════════════

SUPABASE_URL = "https://pqtjvdfcitdpveuqzgpk.supabase.co"
SUPABASE_SERVICE_ROLE_KEY = ""  # ⚠️ Paste service role key here (NEVER share)

USER_ID = ""  # ⚠️ Paste target user's UUID here

# ══════════════════════════════════════════════

if not SUPABASE_SERVICE_ROLE_KEY:
    raise ValueError("⛔ SUPABASE_SERVICE_ROLE_KEY is required")
if not USER_ID:
    raise ValueError("⛔ USER_ID is required")

# Initialize Supabase client with service role
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

# Constants
AUDIO_BUCKET = "audio_submissions"
MODELS_BUCKET = "trained_models"
SAMPLE_RATE = 16000
N_MFCC = 40
MAX_PAD_LEN = 64  # Pad/truncate to fixed length

print(f"✅ Supabase connected")
print(f"👤 User: {USER_ID}")

## Section B — Initialize Training Status

In [ ]:
def update_training_status(status, progress, error_message=None, model_version=None):
    """Update user_training_status in Supabase."""
    data = {
        "status": status,
        "progress_percentage": progress,
        "updated_at": datetime.utcnow().isoformat(),
    }
    if error_message is not None:
        data["error_message"] = error_message
    if model_version is not None:
        data["model_version"] = model_version

    # Check if status row exists for any name of this user
    existing = supabase.table("user_training_status") \
        .select("id") \
        .eq("user_id", USER_ID) \
        .execute()

    if existing.data:
        supabase.table("user_training_status") \
            .update(data) \
            .eq("user_id", USER_ID) \
            .execute()
    else:
        data["user_id"] = USER_ID
        # Get the first trained name for this user
        names = supabase.table("trained_names") \
            .select("id") \
            .eq("user_id", USER_ID) \
            .limit(1) \
            .execute()
        if names.data:
            data["trained_name_id"] = names.data[0]["id"]
        supabase.table("user_training_status").insert(data).execute()

    print(f"📊 Status: {status} | Progress: {progress}%")


# Set initial status
update_training_status("DOWNLOADING_AUDIO", 10)
print("✅ Training status initialized")

## Section C — Download Audio Dataset

In [ ]:
# Fetch all trained names for this user
names_response = supabase.table("trained_names") \
    .select("id, name_label") \
    .eq("user_id", USER_ID) \
    .execute()

trained_names = names_response.data
if not trained_names:
    update_training_status("FAILED", 0, error_message="No trained names found for user")
    raise ValueError("⛔ No trained names found")

print(f"📋 Found {len(trained_names)} names:")
for n in trained_names:
    print(f"   • {n['name_label']} ({n['id'][:8]}...)")

In [ ]:
import shutil

# Clean previous dataset
DATASET_DIR = "./dataset"
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

total_samples = 0

for name_entry in trained_names:
    name_id = name_entry["id"]
    name_label = name_entry["name_label"]
    name_dir = os.path.join(DATASET_DIR, name_label)
    os.makedirs(name_dir, exist_ok=True)

    # List all audio files for this name
    storage_path = f"{USER_ID}/{name_id}"
    try:
        files = supabase.storage.from_(AUDIO_BUCKET).list(storage_path)
    except Exception as e:
        print(f"   ⚠️  Could not list files for {name_label}: {e}")
        continue

    wav_files = [f for f in files if f["name"].endswith(".wav")]

    for wav in wav_files:
        file_path = f"{storage_path}/{wav['name']}"
        local_path = os.path.join(name_dir, wav["name"])

        try:
            data = supabase.storage.from_(AUDIO_BUCKET).download(file_path)
            with open(local_path, "wb") as f:
                f.write(data)
            total_samples += 1
        except Exception as e:
            print(f"   ⚠️  Failed to download {wav['name']}: {e}")

    print(f"   ✅ {name_label}: {len(wav_files)} samples downloaded")

print(f"\n📦 Total samples downloaded: {total_samples}")

if total_samples == 0:
    update_training_status("FAILED", 0, error_message="No audio samples found")
    raise ValueError("⛔ No audio samples found")

update_training_status("DOWNLOADING_AUDIO", 30)

## Section D — Feature Extraction (MFCC)

In [ ]:
def extract_mfcc(file_path, sr=SAMPLE_RATE, n_mfcc=N_MFCC, max_pad_len=MAX_PAD_LEN):
    """Extract MFCC features from a WAV file."""
    try:
        audio, _ = librosa.load(file_path, sr=sr)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        # Pad or truncate to fixed length
        if mfcc.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad_len]

        return mfcc
    except Exception as e:
        print(f"   ⚠️  Error processing {file_path}: {e}")
        return None


# Process all samples
features = []
labels = []
skipped = 0

for name_entry in trained_names:
    name_label = name_entry["name_label"]
    name_dir = os.path.join(DATASET_DIR, name_label)

    if not os.path.exists(name_dir):
        continue

    for wav_file in sorted(os.listdir(name_dir)):
        if not wav_file.endswith(".wav"):
            continue

        mfcc = extract_mfcc(os.path.join(name_dir, wav_file))
        if mfcc is not None:
            features.append(mfcc)
            labels.append(name_label)
        else:
            skipped += 1

X = np.array(features)
print(f"\n🔬 Feature extraction complete:")
print(f"   Shape: {X.shape}")
print(f"   Labels: {len(set(labels))} classes")
print(f"   Samples: {len(labels)} (skipped: {skipped})")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

print(f"   Classes: {list(label_encoder.classes_)}")

# Normalize features
X = X / np.max(np.abs(X))

# Reshape for Dense layers: (samples, n_mfcc * max_pad_len)
X_flat = X.reshape(X.shape[0], -1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"   Train: {len(X_train)} | Test: {len(X_test)}")

update_training_status("TRAINING", 40)

## Section E — Model Training

In [ ]:
# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_flat.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# Custom callback to update training progress in Supabase
class TrainingProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        # Map epoch progress from 40% to 80%
        progress = 40 + int((epoch + 1) / self.total_epochs * 40)
        progress = min(progress, 80)

        acc = logs.get('accuracy', 0)
        val_acc = logs.get('val_accuracy', 0)
        print(f"   Epoch {epoch+1}/{self.total_epochs} — acc: {acc:.4f} | val_acc: {val_acc:.4f} | progress: {progress}%")

        # Update every 5 epochs to avoid rate limiting
        if (epoch + 1) % 5 == 0 or (epoch + 1) == self.total_epochs:
            try:
                update_training_status("TRAINING", progress)
            except Exception as e:
                print(f"   ⚠️  Status update failed: {e}")


EPOCHS = 50
BATCH_SIZE = 16

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[TrainingProgressCallback(EPOCHS)],
    verbose=0,
)

# Final evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n🎯 Final Results:")
print(f"   Loss:     {loss:.4f}")
print(f"   Accuracy: {accuracy:.4f}")

## Section F — Convert to TFLite

In [ ]:
update_training_status("UPLOADING_MODEL", 85)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save locally
TFLITE_PATH = "./vibro_model.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

model_size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f"✅ TFLite model saved: {model_size_kb:.1f} KB")

# Save label mapping
label_mapping = {int(i): label for i, label in enumerate(label_encoder.classes_)}
LABELS_PATH = "./label_mapping.json"
with open(LABELS_PATH, "w") as f:
    json.dump(label_mapping, f, indent=2)

print(f"✅ Label mapping saved: {label_mapping}")

## Section G — Upload Model to Supabase

In [ ]:
try:
    # Determine next model version
    existing_models = supabase.table("trained_models") \
        .select("model_version") \
        .eq("user_id", USER_ID) \
        .order("model_version", desc=True) \
        .limit(1) \
        .execute()

    if existing_models.data:
        next_version = existing_models.data[0]["model_version"] + 1
    else:
        next_version = 1

    print(f"📦 Model version: v{next_version}")

    # Upload TFLite model
    model_remote_path = f"{USER_ID}/model_v{next_version}.tflite"

    with open(TFLITE_PATH, "rb") as f:
        supabase.storage.from_(MODELS_BUCKET).upload(
            model_remote_path,
            f.read(),
            file_options={"content-type": "application/octet-stream", "upsert": "true"}
        )

    print(f"✅ Model uploaded: {model_remote_path}")

    # Upload label mapping
    labels_remote_path = f"{USER_ID}/labels_v{next_version}.json"

    with open(LABELS_PATH, "rb") as f:
        supabase.storage.from_(MODELS_BUCKET).upload(
            labels_remote_path,
            f.read(),
            file_options={"content-type": "application/octet-stream", "upsert": "true"}
        )

    print(f"✅ Labels uploaded: {labels_remote_path}")

    # Get the first trained_name_id for the DB record
    first_name_id = trained_names[0]["id"]

    # Insert into trained_models table
    supabase.table("trained_models").insert({
        "user_id": USER_ID,
        "trained_name_id": first_name_id,
        "model_version": next_version,
        "model_path": model_remote_path,
        "training_sample_count": total_samples,
        "accuracy_metric": float(accuracy),
    }).execute()

    print(f"✅ Model record inserted in trained_models table")

    # Update training status to COMPLETED
    update_training_status("COMPLETED", 100, model_version=next_version)

    print(f"\n══════════════════════════════════════")
    print(f"🎉 TRAINING COMPLETE")
    print(f"   User:     {USER_ID}")
    print(f"   Version:  v{next_version}")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Size:     {model_size_kb:.1f} KB")
    print(f"   Names:    {list(label_encoder.classes_)}")
    print(f"══════════════════════════════════════")

except Exception as e:
    error_msg = str(e)
    print(f"\n⛔ UPLOAD FAILED: {error_msg}")
    update_training_status("FAILED", 0, error_message=error_msg)